# Speaker Diarization Lab — Failure-Driven Notebook

**Course:** IMA4511 — Pattern Recognition and Biometrics  
**Topic:** Speaker diarization: *who spoke when?*  
**Estimated duration:** 2h30–3h  
**Submission:** a zip file containing your report, your code, and your audio files.

This lab is designed around **empirical observations**. Generic explanations are not enough. Most answers must refer to your own recordings, timestamps, plots, diarization outputs, and parameter sweeps.

## Learning objectives

By the end of the lab, you should be able to:

1. record or load short multi-speaker audio;
2. visualize speech signals with waveforms and spectrograms;
3. implement a simple energy-based voice activity detector;
4. run a pretrained diarization model;
5. inspect diarization errors at the timestamp level;
6. understand failure modes caused by noise, overlap, speaker count, and short turns.


## 0. Setup

This notebook runs both locally and on **Google Colab**.

On Colab, run the installation cell below first (it takes a few minutes). Locally, run it only if the packages are missing.

`pyannote.audio` requires a Hugging Face account, acceptance of the model terms for `pyannote/speaker-diarization-3.1` (and `pyannote/segmentation-3.0`), and an access token with read permission. The token is needed to run the diarization pipeline in Section 6.

In [ ]:
# Run this cell once.
# On Google Colab (or a fresh environment) it installs the packages needed for diarization.
# numpy, pandas, matplotlib, librosa, soundfile and scikit-learn are already available on Colab.
# We pin numpy<2.1 so that librosa/numba keep working with the diarization stack.
%pip install -q "numpy<2.1" pyannote.audio pyannote.metrics
# Red "dependency resolver" warnings about colab/gradio/opentelemetry are harmless.
# If Colab shows a "restart session" prompt, do it (Runtime -> Restart session),
# then re-run from Section 1 (you do not need to run this install cell again).

In [ ]:
from pathlib import Path
import json
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display
import soundfile as sf

from IPython.display import Audio, display, Markdown

LAB_DIR = Path("diarization_lab_artifacts")
LAB_DIR.mkdir(exist_ok=True)

print(f"Artifacts will be saved in: {LAB_DIR.resolve()}")

## 1. Recording protocol

Work in groups of 2–3 students. You must create three short recordings.

Use your phone, laptop microphone, or any recording tool, then upload/save the files into the notebook folder.

Required files:

| File | Duration | Content |
|---|---:|---|
| `clean.wav` | 25–40 s | 2 speakers alternate, no intentional overlap |
| `overlap.wav` | 20–35 s | same speakers, with a clear overlap region |
| `degraded.wav` | 20–35 s | same speakers, but harder condition: noise, distance, soft voice, background music, or fast turns |


In [ ]:
# On Google Colab, run this cell to upload your three .wav files into the lab folder.
# Locally, just place clean.wav / overlap.wav / degraded.wav inside LAB_DIR yourself.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Select your .wav files (clean.wav, overlap.wav, degraded.wav)...")
    uploaded = files.upload()
    for fname in uploaded:
        dest = LAB_DIR / fname
        Path(fname).replace(dest)
        print(f"Saved {dest}")
else:
    print(f"Not on Colab. Place your .wav files in: {LAB_DIR.resolve()}")

In [ ]:
# Make sure your three .wav files live in LAB_DIR (use the upload cell above on Colab).
AUDIO_PATHS = {
    "clean": LAB_DIR / "clean.wav",
    "overlap": LAB_DIR / "overlap.wav",
    "degraded": LAB_DIR / "degraded.wav",
}

for name, path in AUDIO_PATHS.items():
    print(f"{name:8s} -> {path} | exists: {path.exists()}")

### Question 1 — Recording metadata

In your report, document the following for each recording (`clean`, `overlap`, `degraded`): number of speakers, recording condition, microphone used, and any relevant notes. Be precise — this information is used later to interpret errors.

## 2. Load and listen to your recordings

This section loads audio at 16 kHz mono. Keep the recordings short. If your files are `.m4a` or `.mp3`, convert them to `.wav` before running the rest of the notebook.


In [ ]:
def load_audio(path, sr=16000):
    y, sr = librosa.load(path, sr=sr, mono=True)
    return y, sr

loaded = {}
for name, path in AUDIO_PATHS.items():
    if path.exists():
        y, sr = load_audio(path)
        loaded[name] = {"audio": y, "sr": sr}
        print(f"{name:8s}: {len(y)/sr:.2f} s, sr={sr}")
    else:
        print(f"Missing: {path}")

In [ ]:
# Listen to a recording.
# Change "clean" to "overlap" or "degraded".
name = "clean"
if name in loaded:
    display(Audio(loaded[name]["audio"], rate=loaded[name]["sr"]))

### Question 2 — Manual speaker timeline

Before running any model, manually annotate the approximate speaker turns of the clean recording (using the audio player and your ears).

In your report, give a table of turns with columns `speaker`, `start`, `end` (in seconds). Use labels `A`, `B`, and optionally `OVERLAP`.

## 3. Waveform and spectrogram visualization

Speaker changes are sometimes visible in the waveform, but not always. The objective here is to inspect the signal before trusting model outputs.


In [ ]:
def plot_waveform_and_spectrogram(y, sr, title="audio"):
    duration = len(y) / sr
    fig = plt.figure(figsize=(14, 3))
    librosa.display.waveshow(y, sr=sr)
    plt.title(f"Waveform — {title} ({duration:.1f}s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.show()

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80, hop_length=256)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig = plt.figure(figsize=(14, 4))
    librosa.display.specshow(mel_db, sr=sr, hop_length=256, x_axis="time", y_axis="mel")
    plt.colorbar(format="%+2.0f dB")
    plt.title(f"Log-mel spectrogram — {title}")
    plt.show()

for name, item in loaded.items():
    plot_waveform_and_spectrogram(item["audio"], item["sr"], title=name)

### Question 3 — Signal-level observations

Answer the following in your report, using timestamps from your own recordings:

1. In the clean recording, give one approximate speaker-change timestamp.
2. Is this change visible in the waveform? Why?
3. In the overlap recording, give the overlap start/end times.
4. In the degraded recording, what makes the signal harder?

## 4. Implement a simple energy-based VAD

Voice Activity Detection (VAD) decides when speech is present. Here, we implement a deliberately simple VAD using frame-level RMS energy.

This method is intentionally weak. You should observe where it fails.


In [ ]:
def energy_vad(y, sr, threshold_percentile=60, frame_ms=30, hop_ms=10):
    frame_length = int(frame_ms / 1000 * sr)
    hop_length = int(hop_ms / 1000 * sr)
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_length)
    threshold = np.percentile(rms, threshold_percentile)
    speech = rms > threshold
    return times, rms, threshold, speech

def plot_vad(y, sr, title, threshold_percentile=60):
    times, rms, threshold, speech = energy_vad(y, sr, threshold_percentile=threshold_percentile)
    plt.figure(figsize=(14, 3))
    plt.plot(times, rms)
    plt.axhline(threshold, linestyle="--")
    plt.fill_between(times, 0, rms.max() if len(rms) else 0, where=speech, alpha=0.25)
    plt.title(f"Energy VAD — {title} — threshold percentile={threshold_percentile}")
    plt.xlabel("Time (s)")
    plt.ylabel("RMS energy")
    plt.show()
    return times, rms, threshold, speech

name = "clean"
if name in loaded:
    _ = plot_vad(loaded[name]["audio"], loaded[name]["sr"], name, threshold_percentile=60)

### Question 4 — VAD threshold sweep

Run the next cell and inspect the plots. Then, in your report, summarize what you observe for the `clean` and `degraded` recordings at a low (40) and a high (80) threshold percentile: the main effect on detected speech and one example timestamp.

Reminder: "missed speech" means speech the VAD marks as non-speech; "false alarm" means non-speech/noise the VAD marks as speech.

In [ ]:
vad_results = []
for rec_name in ["clean", "degraded"]:
    if rec_name not in loaded:
        continue
    y, sr = loaded[rec_name]["audio"], loaded[rec_name]["sr"]
    for p in [40, 50, 60, 70, 80]:
        times, rms, threshold, speech = energy_vad(y, sr, threshold_percentile=p)
        speech_ratio = float(np.mean(speech))
        vad_results.append({
            "recording": rec_name,
            "threshold_percentile": p,
            "threshold_value": threshold,
            "speech_ratio": speech_ratio,
        })
        plot_vad(y, sr, f"{rec_name}", threshold_percentile=p)

vad_results = pd.DataFrame(vad_results)
vad_results.to_csv(LAB_DIR / "vad_sweep.csv", index=False)
vad_results

## 5. Add controlled white noise

This section creates synthetic noisy versions of your clean recording. Unlike real-world noise, white noise lets us control the Signal-to-Noise Ratio (SNR).

Observe whether the first failure comes from VAD, segmentation, or speaker assignment.


In [ ]:
def add_noise_at_snr(clean, snr_db, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(size=len(clean))
    clean_power = np.mean(clean ** 2) + 1e-12
    noise_power = np.mean(noise ** 2) + 1e-12
    target_noise_power = clean_power / (10 ** (snr_db / 10))
    scaled_noise = noise * np.sqrt(target_noise_power / noise_power)
    return clean + scaled_noise

if "clean" in loaded:
    y, sr = loaded["clean"]["audio"], loaded["clean"]["sr"]
    for snr_db in [20, 10, 0]:
        noisy = add_noise_at_snr(y, snr_db)
        noisy = noisy / max(1.0, np.max(np.abs(noisy)))
        out_path = LAB_DIR / f"clean_noise_{snr_db}db.wav"
        sf.write(out_path, noisy, sr)
        print(f"Saved {out_path}")

In [ ]:
# Listen to a noisy version.
path = LAB_DIR / "clean_noise_10db.wav"
if path.exists():
    y_noisy, sr_noisy = load_audio(path)
    display(Audio(y_noisy, rate=sr_noisy))
    plot_waveform_and_spectrogram(y_noisy, sr_noisy, title="clean + 10 dB white noise")
    _ = plot_vad(y_noisy, sr_noisy, "clean + 10 dB white noise", threshold_percentile=60)

### Question 5 — SNR observations

Using the generated noisy files, answer in your report for the clean, 20 dB, 10 dB and 0 dB versions:

1. What changes in the waveform/spectrogram?
2. What changes in the VAD output?
3. Give one timestamped observation.

## 6. Run a pretrained diarization pipeline

This section uses `pyannote.audio`. You need:

1. a Hugging Face account;
2. access accepted for the relevant pyannote models;
3. a token with read permission.

The next cell requires your token and will stop with an error if none is provided.

In [ ]:
# Provide your Hugging Face access token (REQUIRED for the diarization pipeline).
# - Locally: export HF_TOKEN=hf_... before launching, or enter it when prompted.
# - On Colab: add it as a Secret named HF_TOKEN (key icon in the sidebar), or enter it when prompted.
import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = ""

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Enter your Hugging Face token: ").strip()

if not HF_TOKEN:
    raise ValueError(
        "A Hugging Face token is required. Create one at "
        "https://huggingface.co/settings/tokens and accept the model conditions for "
        "pyannote/speaker-diarization-3.1 (and pyannote/segmentation-3.0)."
    )

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN)

try:
    import torch
    if torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
        print("Using GPU.")
except Exception:
    pass

print("Pipeline loaded.")

In [ ]:
def save_rttm(diarization, uri, path):
    with open(path, "w") as f:
        diarization.write_rttm(f)

def diarization_to_dataframe(diarization):
    rows = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        rows.append({
            "speaker": speaker,
            "start": float(turn.start),
            "end": float(turn.end),
            "duration": float(turn.end - turn.start),
        })
    return pd.DataFrame(rows)

diarization_outputs = {}

if pipeline is not None:
    for rec_name, path in AUDIO_PATHS.items():
        if path.exists():
            print(f"Running diarization on {rec_name}...")
            diarization = pipeline(str(path))
            diarization_outputs[rec_name] = diarization
            rttm_path = LAB_DIR / f"{rec_name}.rttm"
            csv_path = LAB_DIR / f"{rec_name}_diarization.csv"
            save_rttm(diarization, rec_name, rttm_path)
            df = diarization_to_dataframe(diarization)
            df.to_csv(csv_path, index=False)
            display(Markdown(f"### {rec_name} diarization"))
            display(df)
            print(f"Saved {rttm_path} and {csv_path}")

## 7. Visualize diarization timelines

This function plots model-predicted speaker regions. Use it to inspect boundary errors, merged speakers, fragmented speakers, and overlap behavior.


In [ ]:
def plot_diarization_df(df, title="diarization"):
    if df is None or len(df) == 0:
        print("Empty diarization dataframe.")
        return
    speakers = sorted(df["speaker"].unique())
    speaker_to_y = {spk: i for i, spk in enumerate(speakers)}

    plt.figure(figsize=(14, 1.5 + 0.4 * len(speakers)))
    for _, row in df.iterrows():
        y = speaker_to_y[row["speaker"]]
        plt.plot([row["start"], row["end"]], [y, y], linewidth=8)
    plt.yticks(range(len(speakers)), speakers)
    plt.xlabel("Time (s)")
    plt.title(title)
    plt.grid(axis="x")
    plt.show()

# Plot saved diarization CSVs if they exist.
for rec_name in ["clean", "overlap", "degraded"]:
    csv_path = LAB_DIR / f"{rec_name}_diarization.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        plot_diarization_df(df, title=f"Predicted diarization — {rec_name}")

### Question 6 — Clean recording: comment on the results

Compare the clean audio, your manual annotation, and the predicted diarization timeline.

In your report, comment on the results you obtained: what does the model get right, and where does it go wrong? Support your comments with timestamps from your own recording.

### Question 7 — Overlap recording: comment on the results

Inspect the region where two people speak at the same time.

In your report, comment on the results you obtained: how does the system behave during overlap? Support your comments with timestamps from your own recording.

### Question 8 — Degraded recording: comment on the results

Use your degraded recording.

In your report, comment on the results you obtained: how robust is the system to the degradation you introduced? Support your comments with timestamps from your own recording.

## 8. Clustering intuition with synthetic embeddings

This section is independent of pyannote. It demonstrates why the number of clusters matters.

We create synthetic “speaker embeddings” for two speakers, then run clustering with the wrong number of clusters.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(42)

# Two synthetic speakers in a higher-dimensional space.
speaker_a = rng.normal(loc=0.0, scale=0.7, size=(20, 16))
speaker_b = rng.normal(loc=3.0, scale=0.7, size=(20, 16))
X = np.vstack([speaker_a, speaker_b])
true_labels = np.array([0] * len(speaker_a) + [1] * len(speaker_b))

X2 = PCA(n_components=2).fit_transform(X)

cluster_sweep = []
for k in [1, 2, 3, 4]:
    pred = AgglomerativeClustering(n_clusters=k).fit_predict(X)
    ari = adjusted_rand_score(true_labels, pred)
    cluster_sweep.append({"n_clusters": k, "adjusted_rand_index": ari})

    plt.figure(figsize=(5, 5))
    plt.scatter(X2[:, 0], X2[:, 1], c=pred)
    plt.title(f"Agglomerative clustering with k={k} | ARI={ari:.2f}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

cluster_sweep = pd.DataFrame(cluster_sweep)
cluster_sweep

### Question 9 — Speaker count constraint

Based on the clustering plots, answer in your report:

1. What happens for k = 1, 2, 3 and 4 clusters?
2. What is under-clustering?
3. What is over-clustering?
4. Why is speaker count a useful but unrealistic input in many real deployments?

## 9. Manual DER-style error decomposition

Diarization Error Rate is usually decomposed into missed speech, false alarm, and speaker confusion.

For this lab, use the simplified definitions:

| Error type | Meaning |
|---|---|
| Missed speech | reference says speech, system says non-speech |
| False alarm | system says speech, reference says non-speech |
| Speaker confusion | system detects speech but assigns the wrong speaker |

We will not require exact collar-based DER computation. The goal is to identify errors correctly.


In [ ]:
# Example toy case.
toy_reference = pd.DataFrame([
    {"speaker": "A", "start": 0.0, "end": 5.0},
    {"speaker": "B", "start": 5.0, "end": 10.0},
])

toy_hypothesis = pd.DataFrame([
    {"speaker": "A", "start": 0.0, "end": 4.0},
    {"speaker": "A", "start": 4.0, "end": 6.0},
    {"speaker": "B", "start": 6.0, "end": 10.0},
])

display(Markdown("### Toy reference"))
display(toy_reference)
display(Markdown("### Toy hypothesis"))
display(toy_hypothesis)

### Question 10 — Error types

In your report:

1. For the toy example above, classify the error type for each interval (0–4 s, 4–5 s, 5–6 s, 6–10 s) and explain it.
2. Identify three errors in your own recordings. For each one, give: recording, start, end, reference speaker(s), predicted speaker(s), error type, and the supporting evidence.

## 10. Submission

Submit a single **zip file** containing:

- your **report** (a separate document with all your answers);
- your **code** (this notebook);
- your **audio files** (`clean.wav`, `overlap.wav`, `degraded.wav`).

In [ ]:
# Helper: bundle your audio files and generated outputs into a single zip.
# On Colab this also triggers a download you can submit.
import shutil

archive_path = shutil.make_archive("diarization_lab_artifacts", "zip", LAB_DIR)
print("Created:", Path(archive_path).resolve())

try:
    from google.colab import files  # type: ignore
    files.download(archive_path)
except Exception:
    print("Download it manually from the path above.")

## Optional open question — Fairness of speaker diarization

This question is optional and open-ended. Answer in your report if you wish.

> How could we evaluate the fairness of speaker diarization? What kinds of biases would you expect to identify, and how could you measure them?